# 웹 크롤링 테스트 노트북

유료 웹사이트에서 입찰 데이터를 수집하는 테스트를 진행합니다.

## 목표
1. 웹사이트 구조 분석
2. 로그인 프로세스 확인
3. 데이터 수집 테스트
4. 수집된 데이터 품질 확인

In [ ]:
import sys
sys.path.append('../src/data')

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns

from web_scraper import BidDataScraper
from data_collector import DataCollector

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("모듈 import 완료")

## 1. 웹사이트 구조 분석

In [ ]:
# 기본 연결 테스트
url = "https://infose.info21c.net/info21c/bids/list?bidtype=con&bid_suc=suc"

try:
    response = requests.get(url)
    print(f"응답 코드: {response.status_code}")
    print(f"응답 헤더: {dict(response.headers)}")
    
    # HTML 구조 미리보기
    soup = BeautifulSoup(response.content, 'html.parser')
    print(f"\n페이지 제목: {soup.title.string if soup.title else 'None'}")
    
    # 로그인 관련 요소 찾기
    login_forms = soup.find_all('form')
    print(f"\n찾은 폼 개수: {len(login_forms)}")
    
except Exception as e:
    print(f"연결 오류: {e}")

## 2. 크롤러 테스트

In [ ]:
# 크롤러 초기화
scraper = BidDataScraper(
    username="wjoon97",
    password="joon3277^^"
)

print("크롤러 초기화 완료")

In [ ]:
# 로그인 테스트
login_success = scraper.login()
print(f"로그인 결과: {login_success}")

In [ ]:
# 데이터 수집 테스트 (1페이지만)
if login_success:
    test_data = scraper.get_page_data(page=1)
    
    if test_data:
        print(f"수집된 데이터 개수: {len(test_data)}")
        print("\n첫 번째 레코드:")
        for key, value in test_data[0].items():
            print(f"  {key}: {value}")
    else:
        print("데이터 수집 실패")
else:
    print("로그인 실패로 데이터 수집 불가")

## 3. 전체 데이터 수집 테스트

In [ ]:
# 여러 페이지 데이터 수집 (3페이지만 테스트)
df_new = scraper.collect_data(max_pages=3, delay=1.0)

if not df_new.empty:
    print(f"총 수집된 레코드: {len(df_new)}")
    print(f"컬럼: {df_new.columns.tolist()}")
    print("\n데이터 타입:")
    print(df_new.dtypes)
    
    print("\n데이터 미리보기:")
    display(df_new.head())
else:
    print("데이터 수집 실패")

## 4. 데이터 품질 분석

In [ ]:
if not df_new.empty:
    print("=== 데이터 품질 분석 ===")
    
    # 기본 통계
    print(f"\n총 레코드 수: {len(df_new)}")
    print(f"컬럼 수: {len(df_new.columns)}")
    
    # 결측값 분석
    missing_values = df_new.isnull().sum()
    print("\n컬럼별 결측값:")
    for col, missing in missing_values.items():
        if missing > 0:
            print(f"  {col}: {missing}개 ({missing/len(df_new)*100:.1f}%)")
    
    # 중복값 확인
    duplicates = df_new.duplicated().sum()
    print(f"\n중복 레코드: {duplicates}개")
    
    # 각 컬럼의 고유값 개수
    print("\n컬럼별 고유값 개수:")
    for col in df_new.columns:
        unique_count = df_new[col].nunique()
        print(f"  {col}: {unique_count}개")
        
        # 카테고리 변수의 경우 값 분포 확인
        if col in ['지역', '업종'] and unique_count < 20:
            print(f"    값 분포: {df_new[col].value_counts().to_dict()}")

## 5. 데이터 시각화

In [ ]:
if not df_new.empty and len(df_new) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('수집된 데이터 분포 분석', fontsize=16)
    
    # 1. 지역별 분포
    if '지역' in df_new.columns:
        df_new['지역'].value_counts().head(10).plot(kind='bar', ax=axes[0,0])
        axes[0,0].set_title('지역별 입찰 건수')
        axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. 업종별 분포
    if '업종' in df_new.columns:
        df_new['업종'].value_counts().head(10).plot(kind='bar', ax=axes[0,1])
        axes[0,1].set_title('업종별 입찰 건수')
        axes[0,1].tick_params(axis='x', rotation=45)
    
    # 3. 하한율 분포 (수치 변환 가능한 경우)
    if '하한율' in df_new.columns:
        try:
            # 하한율 데이터를 수치로 변환
            numeric_col = pd.to_numeric(df_new['하한율'], errors='coerce')
            numeric_col.dropna().hist(bins=20, ax=axes[1,0])
            axes[1,0].set_title('하한율 분포')
        except:
            axes[1,0].text(0.5, 0.5, '하한율 데이터 변환 실패', ha='center', va='center')
    
    # 4. 수집 시간별 분포
    if '수집일시' in df_new.columns:
        df_new['수집일시'] = pd.to_datetime(df_new['수집일시'])
        df_new.set_index('수집일시').resample('H').size().plot(ax=axes[1,1])
        axes[1,1].set_title('시간별 수집량')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("시각화할 데이터가 없습니다.")

## 6. 테스트 데이터 저장

In [ ]:
if not df_new.empty:
    # 테스트 데이터 저장
    test_filename = "web_scraping_test_data.csv"
    saved_path = scraper.save_data(df_new, test_filename)
    
    print(f"테스트 데이터 저장 완료: {saved_path}")
    print(f"저장된 레코드 수: {len(df_new)}")
    
    # 저장된 파일 다시 읽어서 확인
    df_loaded = pd.read_csv(saved_path)
    print(f"\n로드된 데이터 확인: {len(df_loaded)}개 레코드")
    print("로드된 데이터 컬럼:", df_loaded.columns.tolist())
else:
    print("저장할 데이터가 없습니다.")

## 다음 단계

1. **HTML 구조 분석**: 실제 웹사이트의 HTML 구조에 맞게 파싱 로직 수정
2. **로그인 프로세스 최적화**: CSRF 토큰, 세션 관리 등 보완
3. **데이터 매핑 정의**: 웹사이트의 실제 필드명과 우리 모델 변수 매핑
4. **에러 처리 강화**: 네트워크 오류, 구조 변경 등에 대한 대응
5. **스케줄링 설정**: 자동 데이터 수집 주기 설정